### Procesamiento de Lenguaje Natural I
# **Desafío 1**



In [1]:
%pip install numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.


### Vectorización de texto y modelo de clasificación Naïve Bayes con el dataset 20 newsgroups

In [2]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score

Utilizamos **20newsgroups** por ser un dataset clásico de NLP ya viene incluido y formateado en sklearn

In [3]:
from sklearn.datasets import fetch_20newsgroups
import numpy as np

## Carga de datos

Cargamos los datos (ya separados de forma predeterminada en train y test)

El dataset 20 Newsgroups contiene aproximadamente 18 000 publicaciones de grupos de noticias distribuidas en 20 temas. Está dividido en dos subconjuntos: uno para entrenamiento (train set) y otro para pruebas (test set).

In [4]:
fetch_20newsgroups

<function sklearn.datasets._twenty_newsgroups.fetch_20newsgroups(*, data_home=None, subset='train', categories=None, shuffle=True, random_state=42, remove=(), download_if_missing=True, return_X_y=False)>

In [5]:
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

## Vectorización

Instanciamos un vectorizador.

Podemos ver diferentes parámetros de instanciación en la documentación de sklearn https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html

In [6]:
tfidfvect = TfidfVectorizer()

En el atributo `data` accedemos al texto

In [7]:
print(newsgroups_train.data[0])

I was wondering if anyone out there could enlighten me on this car I saw
the other day. It was a 2-door sports car, looked to be from the late 60s/
early 70s. It was called a Bricklin. The doors were really small. In addition,
the front bumper was separate from the rest of the body. This is 
all I know. If anyone can tellme a model name, engine specs, years
of production, where this car is made, history, or whatever info you
have on this funky looking car, please e-mail.


Con la interfaz habitual de sklearn podemos ajustar el vectorizador (obtener el vocabulario y calcular el vector IDF) y transformar directamente los datos.

Podemos denominar `X_train` como la matriz documento-término.

In [8]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)

Recordemos que las vectorizaciones por conteos son de tipo sparse, por ello sklearn convenientemente devuelve los vectores de documentos como matrices de tipo sparse.

In [9]:
print(type(X_train))
print(f'shape: {X_train.shape}')
print(f'Cantidad de documentos: {X_train.shape[0]}')
print(f'Tamaño del vocabulario (dimensionalidad de los vectores): {X_train.shape[1]}')

<class 'scipy.sparse._csr.csr_matrix'>
shape: (11314, 101631)
Cantidad de documentos: 11314
Tamaño del vocabulario (dimensionalidad de los vectores): 101631


Una vez ajustado el vectorizador, podemos acceder a atributos como el vocabulario aprendido. Es un diccionario que va de términos a índices.

El índice es la posición en el vector de documento.

In [10]:
tfidfvect.vocabulary_['car']

25775

Probamos con una palbra que no está en el documento.

In [11]:
#tfidfvect.vocabulary_['Bricklin']

Es muy útil tener el diccionario opuesto que va de índices a términos

In [12]:
idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

En `y_train` guardamos los targets que son enteros

In [13]:
y_train = newsgroups_train.target
y_train[:10]

array([ 7,  4,  4,  1, 14, 16, 13,  3,  2,  4])

Hay 20 clases correspondientes a los 20 grupos de noticias

In [14]:
print(f'clases {np.unique(newsgroups_test.target)}')
newsgroups_test.target_names

clases [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19]


['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

## Similaridad de documentos

Veamos similaridad de documentos. Tomemos algún documento

In [15]:
idx = 4811
print(newsgroups_train.data[idx])

THE WHITE HOUSE

                  Office of the Press Secretary
                   (Pittsburgh, Pennslyvania)
______________________________________________________________
For Immediate Release                         April 17, 1993     

             
                  RADIO ADDRESS TO THE NATION 
                        BY THE PRESIDENT
             
                Pittsburgh International Airport
                    Pittsburgh, Pennsylvania
             
             
10:06 A.M. EDT
             
             
             THE PRESIDENT:  Good morning.  My voice is coming to
you this morning through the facilities of the oldest radio
station in America, KDKA in Pittsburgh.  I'm visiting the city to
meet personally with citizens here to discuss my plans for jobs,
health care and the economy.  But I wanted first to do my weekly
broadcast with the American people. 
             
             I'm told this station first broadcast in 1920 when
it reported that year's presidential elec

Medimos la similaridad coseno con todos los documentos de train

In [16]:
cossim = cosine_similarity(X_train[idx], X_train)[0]

In [17]:
cossim

array([0.1382319 , 0.1067036 , 0.23029327, ..., 0.12320753, 0.08765353,
       0.04415046])

Podemos ver los valores de similaridad ordenados de mayor a menor

In [18]:
np.sort(cossim)[::-1]

array([1.        , 0.70930477, 0.67474953, ..., 0.        , 0.        ,
       0.        ])

Después vemos a qué documentos corresponden

In [19]:
np.argsort(cossim)[::-1]

array([ 4811,  6635,  4253, ...,  1534, 10055,  4750])

Obtenemos los 5 documentos más similares:

In [20]:
mostsim = np.argsort(cossim)[::-1][1:6]
print(mostsim)

[6635 4253 3596 4271 3746]


El documento original pertenece a la clase:

In [21]:
newsgroups_train.target_names[y_train[idx]]

'talk.politics.misc'

Revisamos las clases de los 5 más similares:

In [22]:
for i in mostsim:
  print(newsgroups_train.target_names[y_train[i]])

talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc
talk.politics.misc


### Modelo de clasificación Naïve Bayes

Instanciamos el modelo de clasificación Naive Bayes y lo entrenamos con sklearn

In [23]:
clf = MultinomialNB()
clf.fit(X_train, y_train)

MultinomialNB()

Ya tenemos nuestro vectorizador ya ajustado en train, vectorizamos los textos
del conjunto de test.

In [24]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target
y_pred =  clf.predict(X_test)

El F1-score es una métrica adecuada para evaluar el desempeño de modelos de clasificación, especialmente cuando existe desbalance entre clases.

* El promediado macro calcula el promedio del F1-score de cada clase, otorgando el mismo peso a todas las clases.
* El promediado micro calcula las métricas de forma global considerando todas las predicciones; en problemas de clasificación multiclase suele ser equivalente a la accuracy, por lo que no es la mejor métrica cuando el dataset está desbalanceado.

In [25]:
f1_score(y_test, y_pred, average='macro')

0.5854345727938506

---

## **Consigna del Desafío 1**
**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**



**1. Vectorizar documentos**
* Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

In [28]:
X_train = tfidfvect.fit_transform(newsgroups_train.data)
y_train = newsgroups_train.target

np.random.seed(42)
indices_azar = np.random.choice(len(newsgroups_train.data), size=5, replace=False)

for idx in indices_azar:
    print(f"\n======================================================================")
    print(f"DOCUMENTO BASE SELECCIONADO (Índice: {idx})")
    print(f"======================================================================")
    print(f"Etiqueta Real : {newsgroups_train.target_names[y_train[idx]]}")
    texto_resumido = " ".join(newsgroups_train.data[idx].split()[:40])
    print(f"Contenido (Primeras palabras): {texto_resumido}...\n")
    
    cossim = cosine_similarity(X_train[idx], X_train)[0]
    
    top_5_indices = np.argsort(cossim)[::-1][1:6]
    
    print("--- 5 DOCUMENTOS MÁS SIMILARES ENCONTRADOS ---")
    for rank, sim_idx in enumerate(top_5_indices, start=1):
        score = cossim[sim_idx]
        label_simil = newsgroups_train.target_names[y_train[sim_idx]]
        texto_simil_resumido = " ".join(newsgroups_train.data[sim_idx].split()[:25])
        
        print(f"{rank}. [Similitud Coseno: {score:.4f}] -> Categoría: {label_simil}")
        print(f"   Texto: {texto_simil_resumido}...")


DOCUMENTO BASE SELECCIONADO (Índice: 7492)
Etiqueta Real : comp.sys.mac.hardware
Contenido (Primeras palabras): Could someone please post any info on these systems. Thanks. BoB -- ---------------------------------------------------------------------- Robert Novitskey | "Pursuing women is similar to banging one's head rrn@po.cwru.edu | against a wall...with less opportunity for reward"...

--- 5 DOCUMENTOS MÁS SIMILARES ENCONTRADOS ---
1. [Similitud Coseno: 0.6665] -> Categoría: comp.sys.mac.hardware
   Texto: Hey everybody: I want to buy a mac and I want to get a good price...who doesn't? So, could anyone out there who has found...
2. [Similitud Coseno: 0.3476] -> Categoría: comp.sys.ibm.pc.hardware
   Texto: Hay all: Has anyone out there heard of any performance stats on the fabled p24t. I was wondering what it's performance compared to the 486/66...
3. [Similitud Coseno: 0.1799] -> Categoría: comp.sys.mac.hardware
   Texto: Could someone please send instructions for installing simms

## resultados


**2. Construir un modelo de clasificación por prototipos (tipo zero-shot).**
* Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.



In [31]:
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

print(f"Dimensiones de la matriz de entrenamiento: {X_train.shape}")
print(f"Dimensiones de la matriz de test: {X_test.shape}\n")

print("Calculando matriz de similitudes coseno...")
matriz_similitud = cosine_similarity(X_test, X_train)

indices_max_similitud = np.argmax(matriz_similitud, axis=1)

y_pred_zero_shot = y_train[indices_max_similitud]

print("\n======================================================================")
reporte = classification_report(y_test, y_pred_zero_shot, target_names=newsgroups_test.target_names)
print(reporte)

Dimensiones de la matriz de entrenamiento: (11314, 101631)
Dimensiones de la matriz de test: (7532, 101631)

Calculando matriz de similitudes coseno...

                          precision    recall  f1-score   support

             alt.atheism       0.37      0.51      0.43       319
           comp.graphics       0.54      0.48      0.51       389
 comp.os.ms-windows.misc       0.51      0.46      0.48       394
comp.sys.ibm.pc.hardware       0.52      0.52      0.52       392
   comp.sys.mac.hardware       0.53      0.50      0.52       385
          comp.windows.x       0.70      0.59      0.64       395
            misc.forsale       0.63      0.46      0.53       390
               rec.autos       0.41      0.58      0.48       396
         rec.motorcycles       0.63      0.52      0.57       398
      rec.sport.baseball       0.65      0.54      0.59       397
        rec.sport.hockey       0.75      0.72      0.73       399
               sci.crypt       0.55      0.59      0.5

**3. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación**

* F1-Score Macro en el conjunto de datos de test. Considerar cambiar parámetros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.


In [37]:
vectorizador = TfidfVectorizer(max_df=0.80, min_df=3, sublinear_tf=True, stop_words='english')

X_train_v = vectorizador.fit_transform(newsgroups_train.data)
X_test_v = vectorizador.transform(newsgroups_test.data)

print("--- EVALUACIÓN MODELO 1: MULTINOMIAL NAIVE BAYES ---")
modelo_multinomial = MultinomialNB(alpha=0.1) # alpha bajo para evitar sobre-suavizado
modelo_multinomial.fit(X_train_v, newsgroups_train.target)
y_pred_multi = modelo_multinomial.predict(X_test_v)

print(classification_report(newsgroups_test.target, y_pred_multi, target_names=newsgroups_test.target_names))

print("\n" + "="*70 + "\n")

print("--- EVALUACIÓN MODELO 2: COMPLEMENT NAIVE BAYES ---")
modelo_complement = ComplementNB(alpha=0.1)
modelo_complement.fit(X_train_v, newsgroups_train.target)
y_pred_comple = modelo_complement.predict(X_test_v)

print(classification_report(newsgroups_test.target, y_pred_comple, target_names=newsgroups_test.target_names))

--- EVALUACIÓN MODELO 1: MULTINOMIAL NAIVE BAYES ---
                          precision    recall  f1-score   support

             alt.atheism       0.58      0.42      0.49       319
           comp.graphics       0.63      0.72      0.67       389
 comp.os.ms-windows.misc       0.69      0.52      0.59       394
comp.sys.ibm.pc.hardware       0.61      0.72      0.66       392
   comp.sys.mac.hardware       0.73      0.71      0.72       385
          comp.windows.x       0.80      0.77      0.79       395
            misc.forsale       0.81      0.76      0.79       390
               rec.autos       0.78      0.73      0.76       396
         rec.motorcycles       0.79      0.75      0.77       398
      rec.sport.baseball       0.93      0.82      0.87       397
        rec.sport.hockey       0.59      0.93      0.72       399
               sci.crypt       0.74      0.76      0.75       396
         sci.electronics       0.71      0.58      0.64       393
                 sci.m

**4. Transponer la matriz documento-término.**
* De esa manera se obtiene una matriz término-documento que puede ser interpretada como una colección de vectorización de palabras.
* Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.

In [38]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

X_terms = X_train.T
vocabulario = tfidfvect.get_feature_names_out()

palabras_interes = ["space", "apple", "god", "encryption", "window"]

for palabra in palabras_interes:
    idx = tfidfvect.vocabulary_[palabra]
    
    similitudes = cosine_similarity(X_terms[idx], X_terms)[0]
    
    top_5_idx = np.argsort(similitudes)[::-1][1:6]
    
    print(f"\nPalabra: '{palabra.upper()}' -> Más similares:")
    for rank, sim_idx in enumerate(top_5_idx, start=1):
        print(f"  {rank}. [{similitudes[sim_idx]:.4f}] '{vocabulario[sim_idx]}'")


Palabra: 'SPACE' -> Más similares:
  1. [0.3304] 'nasa'
  2. [0.2966] 'seds'
  3. [0.2928] 'shuttle'
  4. [0.2803] 'enfant'
  5. [0.2465] 'seti'

Palabra: 'APPLE' -> Más similares:
  1. [0.2224] '_macweek_'
  2. [0.2224] 'undisclosed'
  3. [0.2224] '5411'
  4. [0.2224] 'f83y'
  5. [0.2224] 'mb13831fc25'

Palabra: 'GOD' -> Más similares:
  1. [0.2688] 'jesus'
  2. [0.2616] 'bible'
  3. [0.2560] 'that'
  4. [0.2548] 'existence'
  5. [0.2511] 'christ'

Palabra: 'ENCRYPTION' -> Más similares:
  1. [0.3397] 'microcircuit'
  2. [0.3379] 'accommodates'
  3. [0.3379] 'heyman'
  4. [0.3379] 'torrance'
  5. [0.3379] 'nistnews'

Palabra: 'WINDOW' -> Más similares:
  1. [0.3096] 'decoration'
  2. [0.3079] 'manager'
  3. [0.3057] 'xmapwindow'
  4. [0.2837] 'xcreatewindow'
  5. [0.2790] 'xquerytree'
